In [90]:
import pandas as pd
import multiprocessing as mp
from ete3 import Tree
import pickle
from ete3 import NCBITaxa, Tree

In [2]:
import pickle
with open('outputs/connected_components.pickle', 'rb') as f:
    connected_components = pickle.load(f)
with open('outputs/connected_components_reps.pickle', 'rb') as f:
    representative = pickle.load(f)

In [3]:

def init_worker():
    global ncbi
    ncbi = NCBITaxa()
ncbi = NCBITaxa()


def get_taxid(species_name, prefer_rank="species"):
    try:
        name2taxids = ncbi.get_name_translator([species_name])
        if species_name not in name2taxids:
            if len(species_name.split(" "))>2:
                species_name=" ".join(species_name.split(" ")[0:2])
                get_ncbi_lineage_by_rank(" ".join(species_name.split(" ")[0:2]))
                try: 
                    return get_ncbi_lineage_by_rank(" ".join(species_name.split(" ")[0:2]))
                except:
                    print(species_name)
                    return 'None','None'
            raise ValueError(f"No NCBI taxid found for: {species_name!r}")
    
        candidate_taxids = name2taxids[species_name]
    
        if len(candidate_taxids) > 1:
            candidate_ranks = ncbi.get_rank(candidate_taxids)
            preferred = [
                taxid for taxid in candidate_taxids
                if candidate_ranks.get(taxid) == prefer_rank
            ]
            taxid = preferred[0] if preferred else candidate_taxids[0]
        else:
            taxid = candidate_taxids[0]
        
            return taxid
    except:
        print(species_name)

In [91]:

alldf=pd.read_csv("outputs/per_protein_diamond_data.tsv",sep="\t",index_col=0)
dipt_proteins=set(alldf[alldf.slineage.astype(str).str.contains('Diptera')]['accession'])
htgdf=alldf[alldf.HGT]
htgdf=htgdf.set_index('accession')
htgs=set(htgdf.index)

import pickle
with open('outputs/connected_components.pickle', 'rb') as f:
    connected_components = pickle.load(f)
with open('outputs/connected_components_reps.pickle', 'rb') as f:
    representative = pickle.load(f)

In [5]:
def get_sister_info(ranks):
    domain = kingdom = phylum = classt = order = None
    try:
        if 'Viruses' in str(ranks):
            domain='Viruses'
        else:
            domain=ranks['domain']
    except:
        superkingdom='None'
    try:
        kingdom=ranks['kingdom']
    except:
        kingdom='None'
    try:
        phylum=ranks['phylum']
    except:
        phylum='None'
    try:
        classt=ranks['class']
    except:
        classt='None'
    try:
        order=ranks['order']
    except:
        order='None'
    return [domain, kingdom, phylum, classt, order]


In [6]:
import os
tdo=[x for x in os.listdir("outputs/phylogenetic_dataset") if "rev_aa.contree" in os.listdir(f"outputs/phylogenetic_dataset/{x}")]  

In [7]:
def get_sister_names(i):
    try:
        t = Tree(f"outputs/phylogenetic_dataset/{i}/rev_aa_midpoint.treefile")
        
        with open(f"outputs/phylogenetic_dataset/{i}/phylo_dict.pkl", "rb") as f:
            lmap = pickle.load(f)
        
        
            
        for leaf in t:
            if leaf.name in lmap:
                l=lmap[leaf.name]
                leaf.add_features(sciarid= "Sciaridae" in str(l))
        
        mono=[]
        for node in t.get_monophyletic(values=[True], target_attr="sciarid"):
            mono.append(node)
        accepted_nodes=[]
        rejected_nodes=[]
        sister_names=set()
        for node in mono:
            accept = False
            sisters=[x for x in node.up if x.sciarid==False]
            for x in sisters:
                if lmap[x.name].split(";")[-1].strip()=='983967' or 'Cyberlindnera jadinii' in lmap[x.name].split(";")[-1].strip() or 'Metabacillus sp.' in lmap[x.name].split(";")[-1].strip():
                    print(x,lmap[x.name])
            sister_names=sister_names|set([lmap[x.name].split(";")[-1].strip() for x in sisters])
        return sister_names
    except:
        return set()

In [8]:
with mp.Pool(processes=19, initializer=init_worker) as pool:
    results = pool.map(get_sister_names, tdo)


--ODV84082.1 cellular organisms; Eukaryota; Opisthokonta; Fungi; Dikarya; Ascomycota; saccharomyceta; Saccharomycotina; Pichiomycetes; Pichiales; Pichiaceae; Ogataea; Ogataea/Candida clade; Ogataea arabinofermentans; 983967

--XP_020069085.1 cellular organisms; Eukaryota; Opisthokonta; Fungi; Dikarya; Ascomycota; saccharomyceta; Saccharomycotina; Saccharomycetes; Phaffomycetales; Phaffomycetaceae; Cyberlindnera; Cyberlindnera jadinii; Cyberlindnera jadinii NRRL Y-1542

--HZH60411.1 cellular organisms; Bacteria; Bacillati; Bacillota; Bacilli; Caryophanales; Bacillaceae; Metabacillus; unclassified Metabacillus; Metabacillus sp.

--HZH60411.1 cellular organisms; Bacteria; Bacillati; Bacillota; Bacilli; Caryophanales; Bacillaceae; Metabacillus; unclassified Metabacillus; Metabacillus sp.

--HZH60411.1 cellular organisms; Bacteria; Bacillati; Bacillota; Bacilli; Caryophanales; Bacillaceae; Metabacillus; unclassified Metabacillus; Metabacillus sp.

--WP_421650290.1 cellular organisms; Bacte

In [9]:
l=list(results)
all_names=list(set.union(*l))
with mp.Pool(processes=60, initializer=init_worker) as pool:
    tid_results = pool.map(get_taxid, all_names)


Metabacillus sp.
983967
Cyberlindnera jadinii


In [10]:
name_map={x:y for x,y in zip(all_names,tid_results)}

In [11]:
name_map['983967']=get_taxid('Ogataea arabinofermentans')
name_map['Metabacillus sp.']=get_taxid('Metabacillus')
name_map['Cyberlindnera jadinii']=get_taxid('Williopsis jadinii')

In [17]:
def get_sister_lcas(i):
        i=int(i)

    # try:
        t = Tree(f"outputs/phylogenetic_dataset/{i}/rev_aa_midpoint.treefile")
        
        with open(f"outputs/phylogenetic_dataset/{i}/phylo_dict.pkl", "rb") as f:
            lmap = pickle.load(f)
        
        
            
        for leaf in t:
            if leaf.name in lmap:
                l=lmap[leaf.name]
                leaf.add_features(sciarid= "Sciaridae" in str(l))
        
        mono=[]
        for node in t.get_monophyletic(values=[True], target_attr="sciarid"):
        
            mono.append(node)
        accepted_nodes=[]
        rejected_nodes=[]
     
        n=1
        outputs=[]
        for node in mono:
            if len(set([x.name for x in node])&connected_components[i])>0:
              
                sisters=[x for x in node.up if x.sciarid==False]
                o=[f"{i}.{n}",str([x.name for x in node]),str(set([x.name for x in node])&set(connected_components[i]))]
                sister_tids=list(set([name_map[lmap[x.name].split(";")[-1].strip()] for x in sisters if lmap[x.name].split(";")[-1].strip() in name_map]))
                sister_tids=[x for x in sister_tids if x!=None]
                top=ncbi.get_topology(sister_tids)
                name=ncbi.get_taxid_translator([top.taxid])[top.taxid]
                o.extend([top.taxid, str(name)])
                l=ncbi.get_lineage(top.taxid)
                
                ranks={ncbi.get_rank([x])[x]:ncbi.get_taxid_translator([x])[x] for x in l}
                r=get_sister_info(ranks)
                o.extend(r)
                n+=1
                outputs.append(o)
        return  outputs
    # except:
    #     print("fail",i)

In [14]:
with mp.Pool(processes=60, initializer=init_worker) as pool:
    tid_results = pool.map(get_sister_lcas, tdo)

fail 918


In [ ]:
##tree 918 has no non-sciarid sequences

In [19]:
dfresult=pd.DataFrame()
for x in tid_results:

    if x !=None:
        for xi in x:
            dfresult.loc[float(xi[0]),['all_seqs','cluster_seq','taxid','name','domain','kingdom','phylum','class','order']]=xi[1:]

In [20]:
dfresult['tree_number']=[str(x).split(".")[0] for x in dfresult.index]
dfresult=dfresult.sort_index()

In [111]:
dfresult.to_csv("outputs/sister_automated_per_clade.tsv",sep='\t')

In [21]:
import ast
dfresult['cluster_seq']=[ast.literal_eval(x) for x in dfresult['cluster_seq']]

In [22]:
dfs=dfresult.explode('cluster_seq')

In [43]:
dfs=dfs[dfs.cluster_seq.isin(alldf[alldf.tree_HTG].accession)]
dfs.loc[dfs[(dfs.domain.astype(str)=='Eukaryota')&(dfs.kingdom.astype(str)=='None')&~(dfs.name.isin(['Eukaryota','Opisthokonta']))].index,'kingdom']='Protist'
dfs.to_csv("outputs/sister_automated_per_sequence.tsv",sep="\t")

In [44]:
from collections import Counter
Counter(dfs.domain)

Counter({'Eukaryota': 1375,
         'Bacteria': 77,
         None: 57,
         'Viruses': 1,
         'Archaea': 7})

In [28]:
1188/1517

0.7831245880026367

In [47]:
Counter(dfs[dfs.domain=='Eukaryota'].kingdom)

Counter({'Metazoa': 1188,
         'Fungi': 88,
         'None': 47,
         'Protist': 32,
         'Viridiplantae': 20})

In [54]:
m=Counter(dfs[dfs.kingdom=='Metazoa'].phylum)
import numpy as np
np.sum([m[x] for x in m if x not in ['Arthropoda','Rotifera','None']])

26

In [33]:
1075/1517

0.7086354647330257

In [63]:
m

Counter({'Arthropoda': 1075,
         'Nematoda': 11,
         'None': 29,
         'Rotifera': 58,
         'Tardigrada': 5,
         'Mollusca': 3,
         'Nemertea': 1,
         'Cnidaria': 2,
         'Echinodermata': 1,
         'Porifera': 1,
         'Chordata': 1,
         'Ctenophora': 1})

In [57]:
Counter(dfs[dfs.phylum=='Arthropoda']['class'])

Counter({'Collembola': 810,
         'None': 21,
         'Arachnida': 148,
         'Pycnogonida': 3,
         'Insecta': 78,
         'Diplopoda': 11,
         'Ostracoda': 3,
         'Branchiopoda': 1})

## DIAMOND top hit distribution

In [4]:
htgdf=alldf[alldf.tree_HTG]

In [27]:
htgdf[(htgdf.sciname.isin(['Beauveria asiatica','Entomophthora muscae','Neoconidiobolus thromboides FSU 785','Mucor hiemalis', 'Cordyceps sp. RAO-2017']))].shape[0]	

10

In [16]:
htgdf[htgdf.slineage.str.contains('Rickettsiales')].shape[0]

16

In [15]:
htgdf[htgdf.slineage.str.contains('Mucoromycota')].shape[0]

33

In [7]:
htgdf[htgdf.slineage.str.contains('Bdelloidea')].shape[0]

75

In [93]:
Counter(htgdf.domain)

Counter({'Eukaryota': 1390, 'Bacteria': 122, 'Viruses': 2, 'Archaea': 3})

In [32]:
htgdf[htgdf.phylum=='Arthropoda'].shape[0]

1095

In [29]:
1241/1517

0.8180619644034278

In [31]:
785/1517

0.5174686882003955

In [94]:
Counter(htgdf[htgdf.domain=='Eukaryota']['kingdom'])

Counter({'Metazoa': 1241, 'Fungi': 103, nan: 34, 'Viridiplantae': 12})

In [95]:
m=Counter(htgdf[htgdf.kingdom=='Metazoa'].phylum)
import numpy as np
np.sum([m[x] for x in m if x not in ['Arthropoda','Rotifera','None']])

71

In [96]:
Counter(htgdf[htgdf.phylum=='Arthropoda']['class'])

Counter({'Collembola': 785,
         'Insecta': 117,
         'Arachnida': 164,
         'Diplopoda': 16,
         'Malacostraca': 2,
         'Branchiopoda': 6,
         'Merostomata': 2,
         'Ostracoda': 2,
         'Chilopoda': 1})

In [35]:
785/1517

0.5174686882003955

In [101]:
m=dict(Counter(htgdf.sciname))
pd.DataFrame.from_dict(m,orient='index').sort_values(0,ascending=False).iloc[0:20,:]

,0
Folsomia candida,461
Allacma fusca,133
Orchesella dallaii,129
Orchesella cincta,61
Halotydeus destructor,59
Oppia nitens,32
Rotaria sp. Silwood1,18
Leptotrombidium deliense,17
Archispirostreptus syriacus,15
Venturia canescens,13
